# Predicting Customer Default Risk to Support Financial Descision Making

**Dataset:** Default of Credit Card Clients from UCI Machine Learning Repository

**Model:** Logistic Regression

**Additional Techniques:** Feature Selection and Handling Imbalanced data will SMOTE

## Research Question

Which customer characteristics are most predictive of default risk, and how can we use our finding to support financial decision-making?

In [18]:
# Import Libraries
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from imblearn.over_sampling import SMOTE

In [19]:

# Loading in data
credit = pd.read_csv("Credit.csv")

# intuitivly rename column
column_names = {
    'Unnamed: 0': 'ID',
    'X1': 'Credit Amount',
    'X2': 'Gender',
    'X3': 'Education',
    'X4': 'Marital Status',
    'X5': 'Age',
    'X6': 'Repayment Status Sept',
    'X7': 'Repayment Status Aug',
    'X8': 'Repayment Status Jul',
    'X9': 'Repayment Status Jun',
    'X10': 'Repayment Status May',
    'X11': 'Repayment Status Apr',
    'X12': 'Bill Amount Sept',
    'X13': 'Bill Amount Aug',
    'X14': 'Bill Amount Jul',
    'X15': 'Bill Amount Jun',
    'X16': 'Bill Amount May',
    'X17': 'Bill Amount Apr',
    'X18': 'Previous Payment Sept',
    'X19': 'Previous Payment Aug',
    'X20': 'Previous Payment Jul',
    'X21': 'Previous Payment Jun',
    'X22': 'Previous Payment May',
    'X23': 'Previous Payment Apr',
    'Y': 'default'
}

credit = credit.rename(columns=column_names)

# Drop first row (old header)
credit = credit.iloc[1:]

# Reset index
credit = credit.reset_index(drop=True)

# Convert everything to numeric
credit = credit.apply(pd.to_numeric)

print(credit.dtypes)





ID                       int64
Credit Amount            int64
Gender                   int64
Education                int64
Marital Status           int64
Age                      int64
Repayment Status Sept    int64
Repayment Status Aug     int64
Repayment Status Jul     int64
Repayment Status Jun     int64
Repayment Status May     int64
Repayment Status Apr     int64
Bill Amount Sept         int64
Bill Amount Aug          int64
Bill Amount Jul          int64
Bill Amount Jun          int64
Bill Amount May          int64
Bill Amount Apr          int64
Previous Payment Sept    int64
Previous Payment Aug     int64
Previous Payment Jul     int64
Previous Payment Jun     int64
Previous Payment May     int64
Previous Payment Apr     int64
default                  int64
dtype: object


In [20]:
# Checking shape and missing/na values
print(credit.shape)
print(credit.isnull().sum())

(30000, 25)
ID                       0
Credit Amount            0
Gender                   0
Education                0
Marital Status           0
Age                      0
Repayment Status Sept    0
Repayment Status Aug     0
Repayment Status Jul     0
Repayment Status Jun     0
Repayment Status May     0
Repayment Status Apr     0
Bill Amount Sept         0
Bill Amount Aug          0
Bill Amount Jul          0
Bill Amount Jun          0
Bill Amount May          0
Bill Amount Apr          0
Previous Payment Sept    0
Previous Payment Aug     0
Previous Payment Jul     0
Previous Payment Jun     0
Previous Payment May     0
Previous Payment Apr     0
default                  0
dtype: int64


In [21]:
# drop ID
credit = credit.drop(columns=["ID"])

# Define X and y
X = credit.drop(columns=["default"])
y = credit["default"]
print(y.value_counts())
print(y.dtype)

default
0    23364
1     6636
Name: count, dtype: int64
int64


In [22]:
print((6636/30000) *100,'%', sep='')

22.12%


In [23]:
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X, y)

selected_mask = selector.get_support()
selected_features = X.columns[selected_mask]

print(selected_features)

Index(['Credit Amount', 'Repayment Status Sept', 'Repayment Status Aug',
       'Repayment Status Jul', 'Repayment Status Jun', 'Repayment Status May',
       'Repayment Status Apr', 'Previous Payment Sept', 'Previous Payment Aug',
       'Previous Payment Jun'],
      dtype='object')


_classif (ANOVA F-test) - Classification (numerical features)

chi2 (Chi-Square test) - Classification (categorical or non-negative features)

mutual_info_classif - Classification (captures non-linear relationships)

f_regression - Regression (continuous target)

In [24]:
X = X[selected_features]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train.shape, X_test.shape)

(24000, 10) (6000, 10)


In [25]:
# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.805
Precision: 0.6780045351473923
Recall: 0.22532027128862095
F1 Score: 0.3382352941176471

Confusion Matrix:
[[4531  142]
 [1028  299]]


TP FN

FP TN


While the model achieves high accuracy, it struggles to identify customers who will default, as shown by the low recall score. This suggests that although predictions of default are relatively precise, many high-risk clients are not captured.




In [26]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.6693333333333333
Precision: 0.3607460788469691
Recall: 0.6412961567445365
F1 Score: 0.46174715138361366

Confusion Matrix:
[[3165 1508]
 [ 476  851]]


When adjusting for class imbalance, the model significantly improves its ability to identify customers at risk of default, increasing recall from 22.5% to 64.1%

From a financial perspective, it is preferable to incorrectly flag a low-risk customer than to miss a high-risk one, as defaults result in direct financial losses.

The results suggest that repayment history variables are the strongest predictors of default risk. While the initial model achieved higher accuracy, it failed to identify most defaulters. After addressing class imbalance, the model became significantly more effective at detecting high-risk customers, making it more suitable for real-world financial decision-making

In [27]:
smote = SMOTE(random_state=2024)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(X_train_smote.shape)
print(y_train_smote.value_counts())

(37382, 10)
default
0    18691
1    18691
Name: count, dtype: int64


In [28]:
# Train SMOTE data
model_smote = LogisticRegression(max_iter=1000)
model_smote.fit(X_train_smote, y_train_smote)

# Predict on OG test set
y_pred_smote = model_smote.predict(X_test_scaled)

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred_smote))
print("Precision:", precision_score(y_test, y_pred_smote))
print("Recall:", recall_score(y_test, y_pred_smote))
print("F1 Score:", f1_score(y_test, y_pred_smote))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))

Accuracy: 0.6626666666666666
Precision: 0.357230643179025
Recall: 0.6571213262999246
F1 Score: 0.46284501061571126

Confusion Matrix:
[[3104 1569]
 [ 455  872]]


In [29]:


# Get coefficients from your model
coefficients = model_smote.coef_[0]

# Get feature names
feature_names = X.columns

# Create dataframe
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Sort by importance
feature_importance = feature_importance.sort_values(by='Coefficient', ascending=False)

# Show results
print(feature_importance)

                 Feature  Coefficient
1  Repayment Status Sept     0.569285
2   Repayment Status Aug     0.086778
3   Repayment Status Jul     0.074304
4   Repayment Status Jun     0.051066
5   Repayment Status May    -0.003361
6   Repayment Status Apr    -0.017744
0          Credit Amount    -0.136226
9   Previous Payment Jun    -0.144596
7  Previous Payment Sept    -0.260408
8   Previous Payment Aug    -0.282113


In [30]:
import matplotlib.pyplot as plt

# Fit SelectKBest
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X, y)

# Get scores
scores = selector.scores_

# Create dataframe
feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'F-Score': scores
})

# Sort
feature_scores = feature_scores.sort_values(by='F-Score', ascending=False)

print(feature_scores)

                 Feature      F-Score
1  Repayment Status Sept  3537.714970
2   Repayment Status Aug  2239.169136
3   Repayment Status Jul  1757.466444
4   Repayment Status Jun  1476.845967
5   Repayment Status May  1304.591176
6   Repayment Status Apr  1085.402485
0          Credit Amount   724.068539
7  Previous Payment Sept   160.403810
8   Previous Payment Aug   103.291524
9   Previous Payment Jun    97.188000


## Conclusion
The results show that repayment history and recent payment behavior are the strongest predictors of default risk, Althoiugh the intial model had the highest accuracy, it preformed poorly at identifying default cases. By apply techniques for unbalanced datasets, like SMOTE, the model became more effective at detectig high risk customers.

This makes the final model more useful for real world financial decsion making, where missing a defultcase a can be more costly than incorrectly flagging a low-risk customer.

The Results from the findings shows customers who had previously missed their credit card payments are most likly to contiune missing their payments. As you can see the repayment coeffiecnts returned the highest F-1 score which indicates that by using featuer selection we were able to find features of customers that miss their payments. There we can conlcude that the demographics of customers were not as telling as others,